<a href="https://colab.research.google.com/github/Thashmikab/london-smart-meter-analysis/blob/main/notebooks/06_who_can_shift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Who can shift? Household types and their response to prices

Notebooks 03–05 treated time-of-use households as one group. This notebook asks which *kinds* of household responded. Households are grouped by the shape of their typical day in July–December 2012, before anyone saw a price, and then each group's response is measured.

**The answers, in plain terms**

- **Five household types emerge from the data:** *evening peakers* (use peaking around 19:00), *home in the daytime* (the largest share of use between 09:00 and 16:00), *early risers* (a 07:00 peak), *late and always-on* (the flattest shape, with a high night-time base), and a small group with *night storage heating* (57% of use between midnight and 06:00).
- **Households at home in the daytime responded most.** They cut use by about **8.6%** during high-price events and are the only type whose overall use fell clearly: **3.9% more than comparable standard households** (95% CI 1.4–6.4%), with a 4.5% cut in evening-peak use. Being at home when an event happens makes it possible to act on it.
- **Evening peakers made most use of cheap periods** (+7.5% during low-price periods), but their overall use didn't change measurably.
- **Late and always-on households responded least.** A flat shape with a high night-time base suggests much of their use is appliances that run constantly, which a price signal doesn't move.
- **Most of the trial's overall effect came from one type.** The 1.8% overall cut found in notebook 03 is close to the size-weighted average of the four groups, and the daytime-at-home group contributes most of it.
- **The night-storage-heating group had no time-of-use households**, so its response can't be assessed.

**What it means:** flexibility depends on routine. Targeting price signals, or support such as automation, at households that are home during the day is likely to deliver more than a blanket tariff; households whose use is mostly always-on appliances need a different lever.

**Inputs:** `halfhourly_cleaned`, `households` and `household_day` from notebooks 01 and 02.

## Setup

In [ ]:
!pip install pyspark

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
#@title Start Spark and load the tables

spark = SparkSession.builder.appName("london-smart-meters-profiles").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/london-smart-meters"

!mkdir -p data/parquet
!cp -r {DRIVE}/halfhourly_cleaned data/parquet/

hh = spark.read.parquet("data/parquet/halfhourly_cleaned")
households = (spark.read.parquet(f"{DRIVE}/households")
              .filter("in_tariff_comparison")
              .select("household_id", "tariff", "acorn_group"))

## 1. Each household's typical day, before the tariff

**How the shapes are made.** For each household in the tariff comparison: its average use in each of the 48 half-hours of the day over July–December 2012 (zero days excluded), divided by its average daily total. Each row then sums to 1, so the groups reflect *when* a household uses electricity, not *how much*; otherwise clustering would mostly separate big homes from small ones. Using 2012 means the groups can't be affected by the tariff.

Result: 5,208 households × 48 half-hours, no missing values; median use 8.5 kWh a day.

In [ ]:
#@title Step 1: average daily shape per household, July-December 2012

profiles_sdf = (hh.filter(~col("zero_day"))
    .withColumn("interval_start", col("timestamp") - F.expr("INTERVAL 30 MINUTES"))
    .filter((F.year("interval_start") == 2012) & (F.month("interval_start") >= 7))
    .join(F.broadcast(households.select("household_id")), "household_id")
    .withColumn("slot", F.hour("interval_start") * 2 + (F.minute("interval_start") / 30).cast("int"))
    .groupBy("household_id").pivot("slot", list(range(48))).agg(F.avg("energy_kwh")))

profiles = profiles_sdf.toPandas().set_index("household_id")
profiles.columns = [int(c) for c in profiles.columns]

daily_kwh = profiles.sum(axis=1)                 # average kWh per day
shape = profiles.div(daily_kwh, axis=0)          # share of the day's use in each half-hour (each row sums to 1)

print(shape.shape)                                # about 5,208 households x 48 half-hours
print("Missing values:", shape.isna().sum().sum())
print(f"Daily kWh: median {daily_kwh.median():.1f}, range {daily_kwh.min():.1f} to {daily_kwh.max():.1f}")

## 2. How many groups?

k-means for 2 to 8 groups. *Within-group spread* always falls with more groups; *silhouette* measures how clearly separated the groups are.

| k | Silhouette | Smallest group |
|---:|---:|---:|
| 2 | 0.689 | 59 |
| 3 | 0.145 | 59 |
| 4 | 0.142 | 54 |
| **5** | **0.143** | 59 |
| 6 | 0.129 | 54 |
| 7–8 | 0.09 | 54 |

k = 2 scores highest only because it separates about 59 homes with an extreme night-time pattern from everyone else. From k = 3 the silhouette is about 0.14, which is normal for load shapes: households form a continuum, so the groups summarise common patterns rather than sharply separate types. **k = 5** keeps the silhouette level and adds a distinct early-morning group that k = 4 misses; k = 6 only splits the evening group into earlier and later versions.

In [ ]:
#@title Step 2a: compare different numbers of groups

X = shape.values
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    rows.append({"k": k,
                 "within-group spread": round(km.inertia_, 3),
                 "silhouette": round(silhouette_score(X, km.labels_, sample_size=3000, random_state=0), 3),
                 "smallest group": np.bincount(km.labels_).min()})
pd.DataFrame(rows)

In [ ]:
#@title Step 2b: what do the groups look like? (try a couple of k values)

def describe_groups(k):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    centres = pd.DataFrame(km.cluster_centers_, columns=range(48))
    windows = {"night 00-06": range(0, 12), "morning 06-09": range(12, 18), "day 09-16": range(18, 32),
               "evening 16-20": range(32, 40), "late 20-24": range(40, 48)}
    summary = pd.DataFrame({name: centres[list(slots)].sum(axis=1) for name, slots in windows.items()}).round(3)
    summary["peak time"] = [f"{s // 2:02d}:{30 * (s % 2):02d}" for s in centres.values.argmax(axis=1)]
    summary["households"] = np.bincount(km.labels_)
    summary["median daily kWh"] = pd.Series(daily_kwh.values).groupby(km.labels_).median().round(1).values
    print(f"\n===== k = {k} =====")
    print(summary)

    plt.figure(figsize=(9, 4))
    for i, row in centres.iterrows():
        plt.plot(np.arange(48) / 2, row.values, label=f"group {i} ({summary.loc[i, 'households']} homes)")
    plt.xticks(range(0, 25, 3)); plt.xlabel("Time of day"); plt.ylabel("Share of daily use per half-hour")
    plt.legend(fontsize=8); plt.title(f"Average daily shape, k = {k}"); plt.show()
    return km

for k in [4, 5, 6]:
    describe_groups(k)

## 3. The five household types

| Type | Homes | Share of use: night / morning / day / evening / late | Peak | Median kWh/day |
|---|---:|---|---|---:|
| Night storage heating | 59 | **57%** / 9% / 13% / 11% / 10% | 00:00 | 15.3 |
| Early risers | 652 | 15% / **18%** / 29% / 23% / 15% | 07:00 | 7.7 |
| Home in the daytime | 1,482 | 11% / 12% / **35%** / 26% / 16% | 17:00 | 8.7 |
| Evening peakers | 1,353 | 12% / 11% / 26% / **28%** / 23% | 19:00 | 8.6 |
| Late and always-on | 1,662 | **19%** / 11% / 28% / 21% / 21% | 20:00 | 8.3 |

The names describe the shapes; the data has no occupations or ages, so they are interpretations. The night-storage-heating shape, with a sharp switch-off at about 06:30, is the signature of storage heaters or hot-water cylinders charging overnight on a timer.

**Balance.** Time-of-use households make up 19–22% of every type except night storage heating, which has none (59 homes where about 12 would be expected, which suggests households on two-rate meters weren't offered the dynamic tariff). The types were formed independently of the tariff, as intended.

**ACORN mix.** Home in the daytime has the highest share of "Adversity" households (38%) and the lowest "Affluent" share (33%); evening peakers are the most affluent type (46%). That fits the shapes: working households peak in the evening, while households at home during the day include more people who are retired, caring or out of work. It's an association, not something the data can confirm for individual homes.

In [ ]:
#@title Step 3: final groups (k = 5), named from their shapes

K = 5
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)
centres = pd.DataFrame(km.cluster_centers_, columns=range(48))
windows = {"night 00-06": range(0, 12), "morning 06-09": range(12, 18), "day 09-16": range(18, 32),
           "evening 16-20": range(32, 40), "late 20-24": range(40, 48)}
share = pd.DataFrame({name: centres[list(slots)].sum(axis=1) for name, slots in windows.items()})

# Name each group by what stands out in its shape
names, remaining = {}, list(range(K))
for window, name in [("night 00-06", "Night storage heating"), ("morning 06-09", "Early risers"),
                     ("day 09-16", "Home in the daytime"), ("evening 16-20", "Evening peakers")]:
    i = share.loc[remaining, window].idxmax()
    names[i] = name
    remaining.remove(i)
names[remaining[0]] = "Late and always-on"

groups = pd.DataFrame({"household_id": shape.index,
                       "group": [names[l] for l in km.labels_],
                       "daily_kwh": daily_kwh.values})
groups = groups.merge(households.toPandas(), on="household_id")

# Size, use, and whether ToU and standard households are spread evenly across groups
summary = groups.groupby("group").agg(households=("household_id", "count"),
                                      tou_households=("tariff", lambda t: (t == "ToU").sum()),
                                      median_daily_kwh=("daily_kwh", "median"))
summary["tou_share"] = (summary["tou_households"] / summary["households"]).round(3)
print(summary)

print("\nACORN mix within each group")
print(pd.crosstab(groups["group"], groups["acorn_group"], normalize="index").round(3))

groups.to_csv(f"{DRIVE}/household_groups.csv", index=False)

## 4. Who responded?

**Event response.** Notebook 05's calculation, done separately for each type: time-of-use against standard households *of the same type* in High, Low and Normal half-hours of 2013, corrected by the same comparison at the same times in 2012.

| Type | Time-of-use homes | High-price events | Low-price periods |
|---|---:|---:|---:|
| **Home in the daytime** | 322 | **−8.6%** | +6.9% |
| Early risers | 133 | −6.7% | +3.3% |
| Evening peakers | 297 | −5.5% | **+7.5%** |
| Late and always-on | 310 | −5.1% | +3.2% |

These are point estimates without confidence intervals, so the differences between types are indicative; the overall comparison below gives the intervals.

In [ ]:
#@title Step 4a: event response by group (2013 vs 2012 placebo)

groups_sdf = spark.createDataFrame(groups[["household_id", "group", "tariff"]])

prof = (hh.filter(~col("zero_day"))
    .withColumn("interval_start", col("timestamp") - F.expr("INTERVAL 30 MINUTES"))
    .filter(F.year("interval_start").isin(2012, 2013))
    .join(F.broadcast(groups_sdf), "household_id")
    .groupBy("interval_start", "group").pivot("tariff", ["Std", "ToU"]).agg(F.avg("energy_kwh"))
    .toPandas())

bands = (spark.read.parquet(f"{DRIVE}/tariffs_2013")
         .select(col("TariffDateTime").alias("interval_start"), col("Tariff").alias("band")).toPandas())
placebo = bands.assign(interval_start=bands["interval_start"] - pd.Timedelta(days=364)).rename(columns={"band": "band_placebo"})
prof = prof.merge(bands, on="interval_start", how="left").merge(placebo, on="interval_start", how="left")

def rel(df, band_col):
    t = df.groupby(band_col)[["Std", "ToU"]].mean()
    r = t["ToU"] / t["Std"]
    return r / r["Normal"]

rows = []
for g, d in prof.groupby("group"):
    r13 = rel(d[d["interval_start"].dt.year == 2013], "band")
    r12 = rel(d[d["interval_start"].dt.year == 2012], "band_placebo")
    rows.append({"group": g,
                 "ToU households": summary.loc[g, "tou_households"],
                 "High-event response %": round(100 * (r13["High"] / r12["High"] - 1), 1),
                 "Low-period response %": round(100 * (r13["Low"] / r12["Low"] - 1), 1)})
event_response = pd.DataFrame(rows).set_index("group")
event_response

**Overall change.** Notebook 03's comparison (July–December 2012 against 2013, time-of-use households' change minus standard households'), for each type, as a percentage of the type's own baseline.

| Type | Daily kWh | 95% CI | p | Evening-peak kWh | 95% CI | p |
|---|---:|---|---:|---:|---|---:|
| **Home in the daytime** | **−3.9%** | −6.4 to −1.4 | 0.003 | **−4.5%** | −7.5 to −1.6 | 0.002 |
| Evening peakers | −1.6% | −4.4 to 1.1 | 0.24 | −1.2% | −4.4 to 2.0 | 0.45 |
| Late and always-on | −1.1% | −5.0 to 2.9 | 0.59 | −2.1% | −6.2 to 2.1 | 0.33 |
| Early risers | +0.1% | −4.9 to 5.0 | 0.98 | +0.1% | −6.2 to 6.3 | 0.99 |

Only the daytime-at-home type changed clearly, and its result holds even after allowing for testing eight comparisons (p = 0.002–0.003, below 0.05 ÷ 8). Weighting the four types by their number of time-of-use households gives about −1.9%, close to the 1.8% overall effect in notebook 03, so most of the trial's effect came from this one type.

In [ ]:
#@title Step 4b: overall change 2012 -> 2013 by group (as in notebook 03)

household_day = spark.read.parquet(f"{DRIVE}/household_day")
sample = (household_day
    .filter(col("in_tariff_comparison") & ~col("zero_day"))
    .filter(col("year").isin(2012, 2013) & (col("month") >= 7))
    .groupBy("household_id", "year").agg(F.avg("kwh").alias("kwh"), F.avg("kwh_peak").alias("peak_kwh"))
    .toPandas())

wide = sample.pivot(index="household_id", columns="year", values=["kwh", "peak_kwh"])
change = pd.DataFrame({"change_kwh": wide[("kwh", 2013)] - wide[("kwh", 2012)],
                       "change_peak_kwh": wide[("peak_kwh", 2013)] - wide[("peak_kwh", 2012)],
                       "base_kwh": wide[("kwh", 2012)],
                       "base_peak_kwh": wide[("peak_kwh", 2012)]}).reset_index()
change = change.merge(groups[["household_id", "group", "tariff"]], on="household_id")

rows = []
for g, d in change.groupby("group"):
    for m in ["kwh", "peak_kwh"]:
        tou = d.loc[d.tariff == "ToU", f"change_{m}"].dropna()
        std = d.loc[d.tariff == "Std", f"change_{m}"].dropna()
        if len(tou) < 30:          # too few ToU households to compare (night storage heating has none)
            continue
        effect = tou.mean() - std.mean()
        se = np.sqrt(tou.var() / len(tou) + std.var() / len(std))
        base = d.loc[d.tariff == "ToU", f"base_{m}"].mean()
        rows.append({"group": g, "measure": "daily kWh" if m == "kwh" else "evening-peak kWh",
                     "effect %": round(100 * effect / base, 1),
                     "95% CI %": f"{100*(effect - 1.96*se)/base:.1f} to {100*(effect + 1.96*se)/base:.1f}",
                     "p-value": round(stats.ttest_ind(tou, std, equal_var=False).pvalue, 3)})
overall = pd.DataFrame(rows)
overall

## 5. The results in one figure

In [ ]:
#@title Figure: who can shift?

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

def make_figure_06(event, overall, path):
    """event: list of (group, n_tou, high_pct, low_pct); overall: dict group -> (eff, lo, hi) for daily kWh."""
    green, red, blue, grey = "#2b6a4f", "#c0392b", "#2f6db3", "#8a8a8a"
    event = sorted(event, key=lambda r: r[2])               # strongest High response first
    groups = [r[0] for r in event]
    ys = np.arange(len(groups))[::-1]
    fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5), sharey=True, gridspec_kw={"width_ratios": [1.15, 1]})

    h = 0.36
    a.barh(ys + h / 2, [r[2] for r in event], height=h, color=red, label="High-price events (use cut)")
    a.barh(ys - h / 2, [r[3] for r in event], height=h, color=blue, label="Low-price periods (use raised)")
    for y, r in zip(ys, event):
        a.text(r[2] - 0.3, y + h / 2, f"{r[2]:+.1f}%", va="center", ha="right", fontsize=9)
        a.text(r[3] + 0.3, y - h / 2, f"{r[3]:+.1f}%", va="center", ha="left", fontsize=9)
    a.axvline(0, color="#444", lw=0.8)
    a.set_yticks(ys, [f"{g}\n({r[1]} time-of-use homes)" for g, r in zip(groups, event)], fontsize=9.5)
    a.set_xlim(-12, 11)
    a.set_xlabel("Change in use during price events (%)")
    a.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
    a.set_title("Response to price events, by household type", loc="left", fontsize=11.5)

    for y, g in zip(ys, groups):
        eff, lo, hi = overall[g]
        sig = not (lo <= 0 <= hi)
        c = green if sig else grey
        b.plot([lo, hi], [y, y], color=c, lw=2.5)
        b.plot(eff, y, "o", color=c, ms=9)
        b.text(hi + 0.3, y, f"{eff:+.1f}%" + ("  (clear effect)" if sig else "  (not distinguishable from 0)"),
               va="center", fontsize=9, color=c)
    b.axvline(0, color="#444", lw=0.8)
    b.set_xlim(-8, 13)
    b.set_xlabel("Extra change in daily use, Jul–Dec 2012 → 2013 (%, with 95% CI)")
    b.set_title("Only households at home in the daytime\nchanged clearly overall", loc="left", fontsize=11.5)
    for ax in (a, b):
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    a.tick_params(axis="y", length=0); b.tick_params(axis="y", length=0)
    fig.suptitle("Who can shift? Households at home in the daytime responded most", x=0.01, ha="left",
                 fontsize=13, y=0.99)
    fig.text(0.01, 0.01, "Groups from k-means clustering of each household's daily shape in Jul–Dec 2012 (before the tariff). "
             "Event responses corrected with a 2012 placebo. The night-storage-heating group (59 homes) had no time-of-use households.",
             fontsize=8, color="#666")
    fig.tight_layout(rect=(0, 0.03, 1, 0.96))
    fig.savefig(path, dpi=160)
    return fig


event = [(g, int(r["ToU households"]), r["High-event response %"], r["Low-period response %"])
         for g, r in event_response.dropna().iterrows()]
daily = overall[overall["measure"] == "daily kWh"].set_index("group")
overall_ci = {g: (r["effect %"], *[float(x) for x in r["95% CI %"].split(" to ")]) for g, r in daily.iterrows()}

!mkdir -p figures
make_figure_06(event, overall_ci, "figures/who_can_shift.png")
plt.show()

## What this means

- **Flexibility depends on routine.** Households that are home during the day responded most to price events and are the only type whose overall use clearly fell. Most of the trial's overall effect came from them.
- **Different types respond to different signals.** Evening peakers took up cheap periods but didn't cut overall; always-on households barely moved, which points to appliances that run regardless of price.
- **For tariff design:** a blanket price signal reaches the households who can respond and many who can't. Targeting, or automation that acts on the household's behalf, is likely to deliver more flexibility per household than price signals alone, especially for evening peakers and always-on homes.

**Limits.** The types come from clustering a continuum of shapes (silhouette about 0.14), so households near a boundary could fall either side. Event responses per type have no confidence intervals. The type names are interpretations of shapes, not facts about who lives in each home. The night-storage-heating type had no time-of-use households, so its flexibility is unknown.